# Solver-Grounded Equation Engine + Teacher CoT — NVIDIA Nemotron Reasoning Challenge (final leaderboard 0.80)

Same teacher chain-of-thought recipe as the 0.83 notebook, with one change: the **equation / cryptarithm** puzzles are solved by our own **deterministic deductive engine** instead of the teacher. For those rows the engine recovers the disguised operator mapping and writes a short, fully *verified* derivation that replaces the (much longer) teacher trace.

It keeps natural reasoning everywhere it helps while making the hardest category exact and compact. See the repository README for the competition rules and a comparison of all three notebooks.

## 0 · Load the data

Load the competition `train.csv` (used only to hold out a small evaluation set) and the teacher chain-of-thought dataset that we train on.

In [ ]:
# Load the competition data + the teacher chain-of-thought dataset.
# train.csv is used only to carve out a held-out eval set; the teacher CoT is the base training signal.

import os
import polars as pl

TRAIN_CANDIDATES = [
    "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv",
    "train.csv",
]
TRAIN_CSV = next((p for p in TRAIN_CANDIDATES if os.path.exists(p)), TRAIN_CANDIDATES[-1])
train = pl.read_csv(TRAIN_CSV)
print(f"Loaded {train.height} train rows from {TRAIN_CSV}")

COT_CANDIDATES = [
    "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv",
    "/kaggle/input/nemotron-cot-tong/problem_ids_matched.csv",
    "winner_dataset/problem_ids_matched.csv",
]
COT_CSV = next((p for p in COT_CANDIDATES if os.path.exists(p)), COT_CANDIDATES[0])
assert os.path.exists(COT_CSV), f"teacher CoT not found; add dgxchen/nemotron-cot-tong. Tried {COT_CANDIDATES}"
print("Teacher CoT path:", COT_CSV)

## 1 · Load the base model and attach a LoRA adapter

Load the 30B base in bf16 and attach a rank-32 LoRA on the attention + MLP projections. The base weights stay frozen.

In [ ]:
# Load Nemotron-3-Nano-30B in bf16 (no quantization) and attach a rank-32 LoRA on the
# attention + MLP projection layers. The base model is frozen; only the adapter is trained.

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel
import torch
import kagglehub
import mamba_ssm

assert mamba_ssm is not None

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
LORA_RANK = 32
MAX_LENGTH = 8192
MODEL_MAX_SEQ = 16384

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MODEL_MAX_SEQ,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    trust_remote_code=True,
    attn_implementation="eager",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Base model loaded via Unsloth (bf16, no quantization).")

TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
           "in_proj", "out_proj", "up_proj", "down_proj"]
print(f"Attaching LoRA adapter (rank={LORA_RANK}, targets={TARGETS})...")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=TARGETS,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

## 2 · Evaluation helpers

Category detection, answer extraction/comparison, and the fixed held-out rows used to sanity-check the adapter.

In [ ]:
# Evaluation helpers: detect category, extract the \boxed{} answer, compare answers
# (exact for binary/string, 1% relative tolerance for numbers), and pick the held-out eval rows.

import re
import math
import random

PROMPT_SUFFIX = ("\nPlease put your final answer inside `\\boxed{}`. "
                 "For example: `\\boxed{your answer}`")


def categorize(prompt: str) -> str:
    """Detect the puzzle category from a keyword in the prompt (used to build EVAL_IDS)."""
    if "bit manipulation" in prompt:   return "bit_manipulation"
    if "gravitational" in prompt:      return "gravity"
    if "unit conversion" in prompt:    return "unit_conversion"
    if "encryption" in prompt:         return "cipher"
    if "numeral system" in prompt:     return "numeral"
    if "equations" in prompt:          return "equation"
    return "unknown"


def compare_answer(stored, predicted) -> bool:
    """Official metric: binary exact, numeric relative-1%, else case-insensitive string.

    Used to verify our equation engine's answer against ground truth before substituting.
    """
    stored = str(stored).strip()
    predicted = str(predicted).strip()
    if re.fullmatch(r"[01]+", stored):
        return predicted.lower() == stored.lower()
    try:
        return math.isclose(float(stored), float(predicted), rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored.lower()


EVAL_PER_CAT = 20
_rng_eval = random.Random(777)
EVAL_IDS = set()
for _cat in ["numeral", "gravity", "unit_conversion", "cipher",
             "bit_manipulation", "equation"]:
    _ids = sorted(r["id"] for r in train.iter_rows(named=True)
                  if categorize(r["prompt"]) == _cat)
    EVAL_IDS.update(_rng_eval.sample(_ids, EVAL_PER_CAT))
print(f"Held out {len(EVAL_IDS)} rows ({EVAL_PER_CAT}/category, seed 777).")


## 3 · Our deductive equation / cryptarithm solver

A deterministic engine that recovers the disguised operator + digit mapping for equation puzzles from the demonstrations and writes a short, fully verified derivation — used to replace the teacher's trace on those rows.

In [ ]:
# Deterministic equation / cryptarithm engine: recover the hidden operator and digit
# mapping from the demonstrated equations and emit a short, verified step-by-step derivation.

# --- equation primitives ---

def _fmt_int_with_dp(value, dp):
    if dp == 0:
        return str(value)
    s = str(value).zfill(dp + 1)
    s = s[: len(s) - dp] + "." + s[len(s) - dp:]
    s = s.lstrip("0") or "0"
    if s.startswith("."):
        s = "0" + s
    return s

def _pad_dp(s, n):
    if "." not in s:
        s = s + "."
    i, f = s.split(".")
    return i + "." + f.ljust(n, "0")

def long_multiplication_lines(a_str, b_str):
    """Step-by-step place-value multiplication; returns (lines, exact_product_str)."""
    a_dp = len(a_str.split(".")[1]) if "." in a_str else 0
    b_dp = len(b_str.split(".")[1]) if "." in b_str else 0
    total_dp = a_dp + b_dp
    a_int = int(a_str.replace(".", ""))
    b_int = int(b_str.replace(".", ""))
    lines, comps = [], []
    b_digits = str(abs(b_int))
    nd = len(b_digits)
    for i in range(nd - 1, -1, -1):
        d = int(b_digits[i])
        if d == 0:
            continue
        comp_scaled = d * (10 ** (nd - 1 - i))
        comp_display = _fmt_int_with_dp(comp_scaled, b_dp)
        if b_dp > 0:
            comp_display = _pad_dp(comp_display, b_dp)
        product_int = a_int * comp_scaled
        product_display = _fmt_int_with_dp(product_int, total_dp)
        if total_dp > 0:
            product_display = _pad_dp(product_display, total_dp)
        comps.append((comp_display, product_int, product_display))
    for comp_display, _, product_display in comps:
        lines.append(f"{a_str} * {comp_display} = {product_display}")
    if len(comps) >= 2:
        running = comps[0][1]
        for i in range(1, len(comps)):
            rd = _fmt_int_with_dp(running, total_dp)
            if total_dp > 0:
                rd = _pad_dp(rd, total_dp)
            running += comps[i][1]
            sd = _fmt_int_with_dp(running, total_dp)
            if total_dp > 0:
                sd = _pad_dp(sd, total_dp)
            lines.append(f"{rd} + {comps[i][2]} = {sd}")
    return lines, _fmt_int_with_dp(a_int * b_int, total_dp)

def _rev_num(s):
    """Reverse a numeric string, keeping a leading minus sign in place."""
    return ("-" + s[1:][::-1]) if s.startswith("-") else s[::-1]

def _num_base(sa, sb):
    """Candidate arithmetic/digit operations on two 2-digit operand strings."""
    a, b = int(sa), int(sb)
    o = [("concatenation", sa + sb), ("reverse concatenation", sb + sa),
         ("addition", str(a + b)), ("absolute difference", str(abs(a - b))),
         ("negated absolute difference", str(-abs(a - b))),
         ("subtraction", str(a - b)), ("reverse subtraction", str(b - a)),
         ("multiplication", str(a * b)), ("multiply+1", str(a * b + 1)),
         ("multiply-1", str(a * b - 1)), ("add+1", str(a + b + 1)),
         ("add-1", str(a + b - 1)), ("sub+1", str(a - b + 1)), ("sub-1", str(a - b - 1))]
    if a and b:
        o.append(("max mod min", str(max(a, b) % min(a, b))))
    if b:
        o += [("integer division", str(a // b)), ("modulo", str(a % b))]
    if a:
        o += [("reverse division", str(b // a)), ("reverse modulo", str(b % a))]
    if len(sa) == 2 and len(sb) == 2:
        d1, d2, d3, d4 = int(sa[0]), int(sa[1]), int(sb[0]), int(sb[1])
        o += [("digit absolute diff", str(abs(d1 - d3)) + str(abs(d2 - d4))),
              ("digit add mod10", str((d1 + d3) % 10) + str((d2 + d4) % 10)),
              ("digit sub mod10", str((d1 - d3) % 10) + str((d2 - d4) % 10)),
              ("cross multiply", str(d1 * d3 + d2 * d4)),
              ("cross multiply rev", str(d1 * d4 + d2 * d3)),
              ("digit multiply", str(d1 * d3) + str(d2 * d4)),
              ("digit multiply rev", str(d1 * d4) + str(d2 * d3)),
              ("digit sum diff", str((d1 + d2) - (d3 + d4))),
              ("digit sum sum", str((d1 + d2) + (d3 + d4))),
              ("digit product diff", str(d1 * d2 - d3 * d4)),
              ("digit product sum", str(d1 * d2 + d3 * d4)),
              ("determinant", str(d1 * d4 - d2 * d3)),
              ("abs determinant", str(abs(d1 * d4 - d2 * d3)))]
    return o

def _num_cands(sa, sb):
    """All numeric candidates, also with reversed operands and/or reversed result."""
    res = {}
    for rop in (0, 1):
        ta, tb = (sa[::-1], sb[::-1]) if rop else (sa, sb)
        for name, val in _num_base(ta, tb):
            for rr in (0, 1):
                v = _rev_num(val) if rr else val
                tag = name + (" [rev operands]" if rop else "") + (" [rev result]" if rr else "")
                res[tag] = v
    return res

_EQ_OP_PRIOR = {op: r for r, op in enumerate([
    "multiplication [rev operands] [rev result]", "addition [rev operands] [rev result]",
    "concatenation", "reverse concatenation", "absolute difference",
    "multiply-1 [rev operands] [rev result]", "multiply+1 [rev operands] [rev result]",
    "add-1 [rev operands] [rev result]", "multiplication", "addition",
    "add+1 [rev operands] [rev result]", "multiply+1", "add+1", "add-1",
    "absolute difference [rev operands] [rev result]", "multiply-1",
    "negated absolute difference [rev operands] [rev result]",
    "max mod min [rev operands] [rev result]", "max mod min",
    "subtraction [rev operands] [rev result]", "negated absolute difference",
    "digit absolute diff", "subtraction",
])}

def _eq_compute_lines(tag, sa, sb):
    """Visible computation for a candidate operation tag on 2-char operand strings.
    Returns (lines, value_str). Mechanical: reversals shown, arithmetic one-line for
    cheap ops, partial products for multiplication."""
    lines = []
    ta, tb = sa, sb
    if "[rev operands]" in tag:
        ta, tb = sa[::-1], sb[::-1]
        lines.append(f"reverse the operands: {sa}->{ta}, {sb}->{tb}")
    base = tag.replace(" [rev operands]", "").replace(" [rev result]", "")
    a, b = int(ta), int(tb)
    if base == "concatenation":
        val = ta + tb
        lines.append(f"write {ta} then {tb}: {val}")
    elif base == "reverse concatenation":
        val = tb + ta
        lines.append(f"write {tb} then {ta}: {val}")
    elif base == "addition":
        val = str(a + b)
        lines.append(f"{ta} + {tb} = {val}")
    elif base == "subtraction":
        val = str(a - b)
        lines.append(f"{ta} - {tb} = {val}")
    elif base == "reverse subtraction":
        val = str(b - a)
        lines.append(f"{tb} - {ta} = {val}")
    elif base == "absolute difference":
        val = str(abs(a - b))
        lines.append(f"|{ta} - {tb}| = {val}")
    elif base == "negated absolute difference":
        val = str(-abs(a - b))
        lines.append(f"-|{ta} - {tb}| = {val}")
    elif base == "multiplication":
        ml, prod = long_multiplication_lines(ta, tb)
        lines += [f"{ta} * {tb}:"] + ["  " + l for l in ml] + [f"  = {prod}"]
        val = prod
    elif base in ("multiply+1", "multiply-1"):
        ml, prod = long_multiplication_lines(ta, tb)
        d = 1 if base.endswith("+1") else -1
        val = str(int(prod) + d)
        lines += [f"{ta} * {tb}:"] + ["  " + l for l in ml]
        lines.append(f"  = {prod}; {prod} {'+' if d > 0 else '-'} 1 = {val}")
    elif base in ("add+1", "add-1"):
        d = 1 if base.endswith("+1") else -1
        val = str(a + b + d)
        lines.append(f"{ta} + {tb} = {a + b}; {a + b} {'+' if d > 0 else '-'} 1 = {val}")
    elif base in ("sub+1", "sub-1"):
        d = 1 if base.endswith("+1") else -1
        val = str(a - b + d)
        lines.append(f"{ta} - {tb} = {a - b}; {a - b} {'+' if d > 0 else '-'} 1 = {val}")
    elif base == "integer division":
        val = str(a // b)
        lines.append(f"{ta} / {tb} = {val} (integer part)")
    elif base == "modulo":
        val = str(a % b)
        lines.append(f"{ta} mod {tb} = {val}")
    elif base == "reverse division":
        val = str(b // a)
        lines.append(f"{tb} / {ta} = {val} (integer part)")
    elif base == "reverse modulo":
        val = str(b % a)
        lines.append(f"{tb} mod {ta} = {val}")
    elif base == "max mod min":
        big, small = max(a, b), min(a, b)
        val = str(big % small)
        lines.append(f"max({ta},{tb}) mod min({ta},{tb}) = {big} mod {small} = {val}")
    else:
        # exotic digit ops: compute via the candidate table, state the digit recipe
        val = _num_cands(sa, sb).get(tag)
        if val is None:
            return None
        lines.append(f"{base} on the digits of {ta} and {tb} gives {val}")
        return lines, val
    if "[rev result]" in tag:
        rev = _rev_num(val)
        lines.append(f"reverse the result: {val} -> {rev}")
        val = rev
    return lines, val


# --- unified equation solver ---

import re
import time

_EQ_TAGS_GLOBAL = None  # filled lazily from _num_cands keys + _EQ_OP_PRIOR

_EQ_CHAR_FAMILY = {
    "+": ["addition [rev operands] [rev result]", "addition", "concatenation",
          "add-1 [rev operands] [rev result]", "add+1 [rev operands] [rev result]",
          "add+1", "add-1", "reverse concatenation"],
    "-": ["absolute difference [rev operands] [rev result]", "absolute difference",
          "negated absolute difference [rev operands] [rev result]",
          "negated absolute difference", "subtraction [rev operands] [rev result]",
          "subtraction", "sub-1", "sub+1"],
    "*": ["multiplication [rev operands] [rev result]", "multiplication",
          "multiply-1 [rev operands] [rev result]", "multiply-1",
          "multiply+1 [rev operands] [rev result]", "multiply+1"],
}
_EQ_LITERAL = {"+": "addition", "-": "subtraction", "*": "multiplication",
               "/": "integer division"}


def _eq_tags_global():
    global _EQ_TAGS_GLOBAL
    if _EQ_TAGS_GLOBAL is None:
        keys = list(_num_cands("12", "34").keys())
        _EQ_TAGS_GLOBAL = sorted(
            keys, key=lambda k: (_EQ_OP_PRIOR.get(k, 99), keys.index(k)))
    return _EQ_TAGS_GLOBAL


def _eq_char_order(opch):
    fam = _EQ_CHAR_FAMILY.get(opch, [])
    return fam + [t for t in _eq_tags_global() if t not in fam]


def _eq_parse(prompt):
    """Parse demo lines/query; each yields variants to absorb the backtick-wrap
    ambiguity (a line may be wrapped in backticks AND backtick is a digit symbol)."""
    lines = []
    for ln in prompt.split("\n"):
        s = ln.strip()
        if " = " not in s or "determine" in s:
            continue
        left, right = s.split(" = ", 1)
        left, right = left.strip(), right.strip()
        lvars = [left]
        if len(left) == 6 and left[0] == "`":
            lvars.append(left[1:])
        if len(left) == 6 and left[-1] == "`":
            lvars.append(left[:-1])
        rvars = [right]
        if len(right) >= 2 and right[-1] == "`":
            rvars.append(right[:-1])
        lv = [L for L in lvars if len(L) == 5]
        if lv:
            lines.append((lv, rvars))
    qm = re.search(r"determine the result for:\s*(.+)", prompt)
    q = qm.group(1).strip() if qm else None
    qvars = []
    if q:
        if len(q) == 5:
            qvars.append(q)
        if len(q) == 6 and q[0] == "`":
            qvars.append(q[1:])
        if len(q) == 6 and q[-1] == "`":
            qvars.append(q[:-1])
    return lines, qvars


def _eq_render_val(val, opch):
    """Result string as the puzzle writes it: minus sign -> the line's op symbol."""
    return (opch + val[1:]) if val.startswith("-") else val


def _mul_one_line(sa, sb):
    """Compact one-line place-value multiplication (for verification lines)."""
    a, b = int(sa), int(sb)
    tens, ones = (b // 10) * 10, b % 10
    parts, tot = [], 0
    for p in (tens, ones):
        if p:
            parts.append(f"{a}*{p}={a * p}")
            tot += a * p
    if len(parts) == 2:
        return f"{', '.join(parts)}, {a * tens}+{a * ones}={tot}"
    return f"{parts[0] if parts else f'{a}*0=0'}"


def _solve_eq_digit(lines_v, qvars):
    """Digit-operand puzzles (identity map): family-A search trace + the sign rule +
    literal reading of operator symbols unseen in the demos."""
    for Q in qvars:
        qa, qop, qb = Q[0:2], Q[2], Q[3:5]
        if not (qa + qb).isdigit():
            continue
        same = []
        for lvars, rvars in lines_v:
            for L in lvars:
                if L[2] == qop and (L[0:2] + L[3:5]).isdigit():
                    same.append((L, rvars))
                    break
        if same:
            order = _eq_char_order(qop)
            winner = None
            for tag in order:
                ok = True
                for L, rvars in same:
                    val = _num_cands(L[0:2], L[3:5]).get(tag)
                    if val is None or _eq_render_val(val, qop) not in rvars:
                        ok = False
                        break
                if ok:
                    winner = tag
                    break
            if winner is None:
                return None
            ans_val = _num_cands(qa, qb).get(winner)
            if ans_val is None:
                return None
            ans = _eq_render_val(ans_val, qop)
            L0, rv0 = same[0]
            a0, b0 = L0[0:2], L0[3:5]
            shown_r = _eq_render_val(_num_cands(a0, b0)[winner], qop)
            lines = [f"Each puzzle assigns one operation to '{qop}'. The lines using "
                     f"'{qop}': " + "; ".join(f"{L} = {_eq_render_val(_num_cands(L[0:2], L[3:5])[winner], qop)}"
                                              for L, _ in same) + ".",
                     f"Test candidate operations on {a0}{qop}{b0} = {shown_r} "
                     f"(most common for '{qop}' first):"]
            shown = 0
            for tag in order:
                if tag == winner:
                    break
                v = _num_cands(a0, b0).get(tag)
                if v is None or shown >= 4:
                    continue
                if _eq_render_val(v, qop) not in rv0:
                    lines.append(f"  {tag}: gives {_eq_render_val(v, qop)}, but the "
                                 f"line says {shown_r} -> no")
                    shown += 1
                else:
                    for L, rvars in same[1:]:
                        v2 = _num_cands(L[0:2], L[3:5]).get(tag)
                        if v2 is None or _eq_render_val(v2, qop) not in rvars:
                            bad = _eq_render_val(v2, qop) if v2 is not None else "nothing"
                            lines.append(f"  {tag}: fits this line but fails {L} "
                                         f"(gives {bad}) -> no")
                            shown += 1
                            break
            comp = _eq_compute_lines(winner, a0, b0)
            if comp is None:
                return None
            wl, wv = comp
            lines.append(f"  {winner}:")
            lines += ["    " + l for l in wl]
            if wv.startswith("-"):
                lines.append(f"    = {wv}; the minus sign is written as the operator "
                             f"symbol: {_eq_render_val(wv, qop)} -> matches.")
            else:
                lines.append(f"    = {wv} -> matches.")
            for L, rvars in same[1:]:
                v = _num_cands(L[0:2], L[3:5])[winner]
                lines.append(f"Check {L}: {winner} gives {_eq_render_val(v, qop)} "
                             f"-> consistent.")
            cq = _eq_compute_lines(winner, qa, qb)
            if cq is None:
                return None
            ql, qv = cq
            lines.append(f"Apply '{winner}' to the query {qa}{qop}{qb}:")
            lines += ["  " + l for l in ql]
            if qv.startswith("-"):
                lines.append(f"  = {qv}, written with '{qop}' as the minus sign: {ans}")
            else:
                lines.append(f"The result is {ans}.")
            return "\n".join(lines), ans
        # operator never demonstrated -> read it literally
        tag = _EQ_LITERAL.get(qop)
        if tag is None:
            return None
        val = _num_cands(qa, qb).get(tag)
        if val is None:
            return None
        ans = _eq_render_val(val, qop)
        cq = _eq_compute_lines(tag, qa, qb)
        if cq is None:
            return None
        ql, qv = cq
        lines = [f"The query's operator '{qop}' never appears in the examples, so it "
                 f"keeps its ordinary meaning: {tag}.",
                 f"Compute {qa}{qop}{qb}:"]
        lines += ["  " + l for l in ql]
        if qv.startswith("-"):
            lines.append(f"  = {qv}, written with '{qop}' as the minus sign: {ans}")
        else:
            lines.append(f"The result is {ans}.")
        return "\n".join(lines), ans
    return None


def _solve_eq_symbol(lines_v, qvars, time_budget=4.0):
    """Symbol-operand puzzles: joint search over (operation per op symbol, injective
    symbol->digit map). DFS, most-constrained line first, per-op-char prior order;
    returns the search path for the trace."""
    t0 = time.time()
    m0 = {}
    for lvars, rvars in lines_v:
        for ch in lvars[0][0:2] + lvars[0][3:5]:
            if ch.isdigit():
                m0[ch] = int(ch)
    for Q in qvars:
        for ch in Q[0:2] + Q[3:5]:
            if ch.isdigit():
                m0[ch] = int(ch)

    def unk_count(lv):
        L = lv[0][0]
        return len({ch for ch in L[0:2] + L[3:5] if ch not in m0})
    order_idx = sorted(range(len(lines_v)),
                       key=lambda i: (unk_count((lines_v[i],)), len(lines_v[i][1][0])))
    sol = {}

    def bt(pos, m, used, opmap, path):
        if sol or time.time() - t0 > time_budget:
            return
        if pos == len(order_idx):
            sol["m"], sol["opmap"], sol["path"] = dict(m), dict(opmap), list(path)
            return
        li = order_idx[pos]
        lvars, rvars = lines_v[li]
        for L in lvars:
            a, opch, b = L[0:2], L[2], L[3:5]
            unk = [ch for ch in dict.fromkeys(a + b) if ch not in m]
            tags = [opmap[opch]] if opch in opmap else _eq_char_order(opch)

            def assign(j):
                if sol or time.time() - t0 > time_budget:
                    return
                if j == len(unk):
                    sa = f"{m[a[0]]}{m[a[1]]}"
                    sb = f"{m[b[0]]}{m[b[1]]}"
                    cands = _num_cands(sa, sb)
                    for tag in tags:
                        val = cands.get(tag)
                        if val is None:
                            continue
                        rend_digits = val[1:] if val.startswith("-") else val
                        neg = val.startswith("-")
                        for R in rvars:
                            rr = R
                            if neg:
                                if len(R) == len(rend_digits) + 1 and R[0] == opch:
                                    rr = R[1:]
                                else:
                                    continue
                            elif len(R) != len(rend_digits):
                                continue
                            m2, used2, ok, fixed = dict(m), set(used), True, []
                            for ch, d in zip(rr, rend_digits):
                                d = int(d)
                                if ch in m2:
                                    if m2[ch] != d:
                                        ok = False
                                        break
                                elif ch.isdigit() or d in used2:
                                    ok = False
                                    break
                                else:
                                    m2[ch] = d
                                    used2.add(d)
                                    fixed.append((ch, d))
                            if ok:
                                op_fixed = [(ch, m2[ch]) for ch in unk] + fixed
                                path.append((li, L, R, tag, val, op_fixed))
                                bt(pos + 1, m2, used2,
                                   dict(opmap, **{opch: tag}), path)
                                if sol:
                                    return
                                path.pop()
                    return
                ch = unk[j]
                for d in range(10):
                    if d in used:
                        continue
                    m[ch] = d
                    used.add(d)
                    assign(j + 1)
                    del m[ch]
                    used.discard(d)
                    if sol:
                        return
            assign(0)
            if sol:
                return
    bt(0, dict(m0), set(m0.values()), {}, [])
    if not sol:
        return None
    m, opmap = sol["m"], sol["opmap"]
    for Q in qvars:
        qa, qop, qb = Q[0:2], Q[2], Q[3:5]
        tag = opmap.get(qop) or _EQ_LITERAL.get(qop)
        if tag is None or any(ch not in m for ch in qa + qb):
            continue
        sa, sb = f"{m[qa[0]]}{m[qa[1]]}", f"{m[qb[0]]}{m[qb[1]]}"
        val = _num_cands(sa, sb).get(tag)
        if val is None:
            continue
        digs = val[1:] if val.startswith("-") else val
        inv = {d: ch for ch, d in m.items()}
        if any(int(d) not in inv for d in digs):
            continue
        enc = "".join(inv[int(d)] for d in digs)
        if val.startswith("-"):
            enc = qop + enc
        return _render_eq_symbol(lines_v, Q, sol, m, opmap, tag, sa, sb, val, enc), enc
    return None


def _render_eq_symbol(lines_v, Q, sol, m, opmap, qtag, sa, sb, qval, ans):
    """Discovery narrative: operator hypotheses tested in prior order, digit anchors
    fixed line by line, full-map verification, then decode-compute-encode."""
    qa, qop, qb = Q[0:2], Q[2], Q[3:5]
    out = ["Here both the digits and the operators are hidden: every symbol stands "
           "for one digit (a fixed substitution) and each operator symbol stands "
           "for one operation.",
           "Equation lines: " + "; ".join(lv[0] + " = " + rv[0]
                                          for lv, rv in lines_v) + "."]
    seen_syms = sorted({ch for lv, rv in lines_v for ch in lv[0][0:2] + lv[0][3:5]
                        if not ch.isdigit()})
    if seen_syms:
        out.append("Digit symbols to decode: " + " ".join(seen_syms) + ".")
    out.append("Work through the lines, most constrained first, trying the usual "
               "meaning of each operator first:")
    for (li, L, R, tag, val, fixed) in sol["path"]:
        a, opch, b = L[0:2], L[2], L[3:5]
        pre = [t for t in _eq_char_order(opch)[:3] if t != tag and
               (opmap.get(opch) == tag)]
        intro = f"Line {L} = {R}: '{opch}' as {tag}"
        if fixed:
            sets = ", ".join(f"{ch}={d}" for ch, d in fixed)
            intro += f" works, taking {sets}"
        sa_l = "".join(str(m[c]) for c in a)
        sb_l = "".join(str(m[c]) for c in b)
        if "multiplication" in tag or "multiply" in tag:
            chk = _mul_one_line(sa_l if "[rev operands]" not in tag else sa_l[::-1],
                                sb_l if "[rev operands]" not in tag else sb_l[::-1])
            intro += f": {chk}"
        intro += f" -> {sa_l}{opch}{sb_l} gives {val}, written '{R}' ✓"
        out.append("  " + intro)
    sym_map = sorted((ch, d) for ch, d in m.items() if not ch.isdigit())
    if sym_map:
        out.append("Digit map found: " + ", ".join(f"{ch}={d}" for ch, d in sym_map)
                   + ".")
    out.append("Verify every line with this map:")
    for lv, rv in lines_v:
        L, R = lv[0], rv[0]
        a, opch, b = L[0:2], L[2], L[3:5]
        tag = opmap[opch]
        sa_l = "".join(str(m[c]) for c in a)
        sb_l = "".join(str(m[c]) for c in b)
        val = _num_cands(sa_l, sb_l)[tag]
        digs = val[1:] if val.startswith("-") else val
        inv = {d: ch for ch, d in m.items()}
        enc = "".join(inv[int(d)] for d in digs)
        if val.startswith("-"):
            enc = opch + enc
        out.append(f"  {L}: {sa_l} {tag} {sb_l} = {val} -> '{enc}' ✓")
    dec = ", ".join(f"{c}={m[c]}" for c in dict.fromkeys(qa + qb))
    out.append(f"Decode the query {qa}{qop}{qb}: {dec} -> {sa}{qop}{sb}, and "
               f"'{qop}' means {qtag}.")
    cq = _eq_compute_lines(qtag, sa, sb)
    if cq is not None:
        ql, _ = cq
        out += ["  " + l for l in ql]
    digs = qval[1:] if qval.startswith("-") else qval
    inv = {d: ch for ch, d in m.items()}
    enc_steps = " ".join(f"{d}->{inv[int(d)]}" for d in digs)
    if qval.startswith("-"):
        out.append(f"= {qval}; encode each digit back through the map ({enc_steps}) "
                   f"and write the minus sign as '{qop}': {ans}")
    else:
        out.append(f"= {qval}; encode each digit back through the map: {enc_steps} "
                   f"-> {ans}")
    return "\n".join(out)


def solve_equation(prompt, time_budget=4.0):
    """Unified equation solver covering the full reverse-engineered generator."""
    lines_v, qvars = _eq_parse(prompt)
    if not lines_v or not qvars:
        return None
    all_digit = all(
        (lv[0][0:2] + lv[0][3:5]).isdigit() for lv, rv in lines_v
    ) and any((Q[0:2] + Q[3:5]).isdigit() for Q in qvars)
    if all_digit:
        return _solve_eq_digit(lines_v, qvars)
    return _solve_eq_symbol(lines_v, qvars, time_budget=time_budget)

## 4 · Build the training set and fine-tune

Use teacher CoT everywhere, except swap in our verified equation derivations where the solver succeeds and round-trips to ground truth; then run one epoch of LoRA SFT.

In [ ]:
# Build the corpus: teacher CoT for every row, but replace each equation/cryptarithm trace
# with our solver's verified derivation when it solves AND round-trips to ground truth.
# Then train one epoch (category-stratified batches) and save the adapter.

import pandas as pd, random, time, re, math
from collections import defaultdict
from datasets import Dataset as HFDataset
from trl import SFTTrainer, SFTConfig
from torch.utils.data import DataLoader, Sampler

SEED = 42
NUM_EPOCHS = 1
EQ_TYPES = ("equation_numeric", "cryptarithm")
EQ_TIME_BUDGET = 0.5

df = pd.read_csv(COT_CSV)
print(f"Teacher CoT rows: {len(df)}")

before = len(df)
df = df[~df["id"].isin(EVAL_IDS)].reset_index(drop=True)
print(f"Held out {before - len(df)} gate rows; training pool: {len(df)}")

print("\nType distribution:")
print(df["type"].value_counts().sort_index())

train_df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)


def boxed_list(s):
    return re.findall(r"\\boxed\{([^}]*)\}", s)


def build_assistant_content(cot, answer):
    """answer-consistency cleaning (the +score fix over naive boxed-stripping).

    The teacher CoT ends '...The answer in \\boxed{-} is \\boxed{<X>}\n</think>\n\\boxed{<X>}'.
    Stripping ALL \\boxed{} then forcing the rounded dataset answer onto a trace that
    concluded a different number teaches 'reasoning != answer'. Instead: cut at </think>,
    keep the teacher's derived <X>; if <X>!=answer but numerically close add one rounding
    sentence; if truly different (teacher error) drop the row.
    """
    boxes = boxed_list(cot)
    answer_str = str(answer).strip()
    if len(boxes) < 2:
        cleaned = re.sub(r"\\boxed\{[^}]*\}", "", cot).rstrip()
        return cleaned + f"\n</think>\n\\boxed{{{answer_str}}}", True
    cot_final = boxes[-1].strip()
    think_close = cot.rfind("</think>")
    body = cot[:think_close].rstrip() if think_close != -1 else cot.rstrip()
    if cot_final == answer_str:
        rounding_note = ""
    else:
        try:
            if abs(float(cot_final) - float(answer_str)) < 1.0:
                rounding_note = (f"\n\nRounding {cot_final} to match the required "
                                 f"precision gives {answer_str}.")
            else:
                return None, False
        except ValueError:
            return None, False
    assistant_content = body + rounding_note + f"\n</think>\n\\boxed{{{answer_str}}}"
    return assistant_content, True


def our_equation_trace(prompt, answer):
    """Our deductive equation/cryptarithm engine. Returns a completion ONLY when it
    solves the puzzle AND the answer round-trips to ground truth, else None. This is the
    Phase-2/3 augmentation: where the teacher GUESSES (equation_numeric_guess) or used the
    weaker concat-only cryptarithm reasoner, our deductive trace replaces it with a
    verified-correct one. Coverage measured on the real rows: eq-deduce 97%, eq-guess 24%,
    cryptarithm-deduce 41% — ~926 rows upgraded from guess/concat to deduction.
    """
    try:
        res = solve_equation(prompt, time_budget=EQ_TIME_BUDGET)
    except Exception:
        return None
    if not res:
        return None
    trace, ans = res
    if not compare_answer(answer, ans):
        return None
    return trace.rstrip() + f"\n</think>\n\\boxed{{{ans}}}"


records = []
record_types = []
n_dropped = 0
n_ours = 0
for _, row in train_df.iterrows():
    prompt = str(row["prompt"])
    answer = str(row["answer"])
    cot = str(row["generated_cot"])
    typ = str(row["type"])

    assistant_content = None
    if typ.startswith(EQ_TYPES):
        assistant_content = our_equation_trace(prompt, answer)
        if assistant_content is not None:
            n_ours += 1

    if assistant_content is None:
        if not cot or cot == "nan" or len(cot.strip()) < 5:
            n_dropped += 1
            continue
        assistant_content, ok = build_assistant_content(cot, answer)
        if not ok:
            n_dropped += 1
            continue

    user_content = prompt + PROMPT_SUFFIX
    records.append({"messages": [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]})
    record_types.append(typ)

dataset = HFDataset.from_list(records)
print(f"\nSFT records: {len(records)} (dropped {n_dropped}); "
      f"our verified equation/cryptarithm traces used: {n_ours}")


def formatting_prompts_func(example):
    messages = example["messages"]
    conversations = [messages] if (messages and isinstance(messages[0], dict)) else messages
    texts = []
    for conversation in conversations:
        try:
            text = tokenizer.apply_chat_template(
                conversation, tokenize=False, add_generation_prompt=False, enable_thinking=True)
        except TypeError:
            text = tokenizer.apply_chat_template(
                conversation, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return texts


training_args = SFTConfig(
    output_dir="/kaggle/working/sft_output",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_length=8192,
    adam_beta1=0.9,
    adam_beta2=0.95,
    adam_epsilon=1e-8,
    weight_decay=0.0,
    max_grad_norm=1e9,
    logging_steps=10,
    disable_tqdm=True,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2,
    remove_unused_columns=False,
    seed=SEED,
    report_to="none",
    packing=False,
)


def build_stratified_index_order(labels, batch_size, seed):
    by_label = defaultdict(list)
    for idx, label in enumerate(labels):
        by_label[label].append(idx)
    rng = random.Random(seed)
    for idx_list in by_label.values():
        rng.shuffle(idx_list)
    n_batches = max(1, math.ceil(len(labels) / batch_size))
    batches = [[] for _ in range(n_batches)]
    batch_order = list(range(n_batches))
    rng.shuffle(batch_order)
    assigned = 0
    for label in sorted(by_label.keys()):
        for idx in by_label[label]:
            batches[batch_order[assigned % n_batches]].append(idx)
            assigned += 1
    return [idx for batch in batches for idx in batch]


class PrecomputedOrderSampler(Sampler):
    def __init__(self, order):
        self.order = list(order)

    def __iter__(self):
        return iter(self.order)

    def __len__(self):
        return len(self.order)


class StratifiedSFTTrainer(SFTTrainer):
    def __init__(self, *args, stratified_order=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.stratified_order = stratified_order

    def get_train_dataloader(self):
        if self.stratified_order is None:
            return super().get_train_dataloader()
        dataloader_kwargs = {
            "batch_size": self.args.per_device_train_batch_size,
            "sampler": PrecomputedOrderSampler(self.stratified_order),
            "collate_fn": self.data_collator,
            "num_workers": self.args.dataloader_num_workers,
            "pin_memory": self.args.dataloader_pin_memory,
            "persistent_workers": self.args.dataloader_persistent_workers,
            "drop_last": self.args.dataloader_drop_last,
        }
        if self.args.dataloader_num_workers > 0:
            dataloader_kwargs["prefetch_factor"] = self.args.dataloader_prefetch_factor
        return DataLoader(self.train_dataset, **dataloader_kwargs)


effective_batch_size = max(1, training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
stratified_order = build_stratified_index_order(record_types, effective_batch_size, SEED)
print(f"Effective batch size: {effective_batch_size}")
print("Type counts:", dict(sorted(pd.Series(record_types).value_counts().to_dict().items())))

trainer = StratifiedSFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    formatting_func=formatting_prompts_func,
    stratified_order=stratified_order,
)

print(">>> TRAINING STARTING <<<")
t0 = time.time()
trainer.train()
print(f"Training done in {(time.time() - t0) / 60:.1f} min")

model.save_pretrained(OUTPUT_DIR)
print("Adapter saved to", OUTPUT_DIR)

## 5 · Package the submission

Zip the adapter weights + config into `submission.zip`.

In [ ]:
# Package the two adapter files (weights + config) into submission.zip.

import os
import json
import subprocess
import zipfile

ADAPTER_CONFIG = os.path.join(OUTPUT_DIR, "adapter_config.json")
ADAPTER_WEIGHTS = os.path.join(OUTPUT_DIR, "adapter_model.safetensors")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")

assert os.path.exists(ADAPTER_CONFIG), "adapter_config.json missing"
assert os.path.exists(ADAPTER_WEIGHTS), "adapter_model.safetensors missing"

with open(ADAPTER_CONFIG) as f:
    cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0
with open(ADAPTER_CONFIG, "w") as f:
    json.dump(cfg, f, indent=2)

if os.path.exists(SUBMISSION_ZIP):
    os.remove(SUBMISSION_ZIP)
subprocess.run(["zip", "-j", SUBMISSION_ZIP, ADAPTER_CONFIG, ADAPTER_WEIGHTS], check=True)

_names = sorted(zipfile.ZipFile(SUBMISSION_ZIP).namelist())
assert _names == ["adapter_config.json", "adapter_model.safetensors"], _names
print("Submission packaged before the gate:", SUBMISSION_ZIP, "->", _names)

## 6 · Clean up the output folder

Reduce the working directory to exactly the submission artifacts.

In [ ]:
# Keep only the submission artefacts (adapter weights + config + submission.zip).

import os
import shutil

KEEP = {"adapter_config.json", "adapter_model.safetensors", "submission.zip"}
for _f in os.listdir(OUTPUT_DIR):
    if _f in KEEP:
        continue
    _p = os.path.join(OUTPUT_DIR, _f)
    (shutil.rmtree if os.path.isdir(_p) else os.remove)(_p)

for _f in sorted(KEEP):
    assert os.path.exists(os.path.join(OUTPUT_DIR, _f)), f"{_f} missing after cleanup"
print("Final /kaggle/working contents:", sorted(os.listdir(OUTPUT_DIR)))